In [1]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [38]:
import sys, pathlib
import pandas as pd
# Add typing import to handle Optional type hint
from typing import Optional, List, Dict, Any  # Import typing module with commonly used types

proj_src = pathlib.Path().resolve().parent.parent / "motec-ld-export-ui" / "motec-to-csv" / "src"
sys.path.append(str(proj_src))
from motec_converter import parse_race_data, to_pandas
sys.path.remove(str(proj_src))
import matplotlib.pyplot as plt
import numpy as np

In [67]:
def process_and_compute_efficiency(dyno_files, motec_files, target_torque=150,
                                   rpm_diff_tol=None, rolling_window=3, plot=True):
    """
    Process dyno and MoTeC data and compute motor efficiency.
    
    Parameters:
    - dyno_df: DataFrame with dyno data indexed by run ID
    - motec_files: list of MoTeC file paths
    - dyno_ids: list of corresponding dyno IDs for each MoTeC file
    - target_torque: torque to filter by (Nm)
    - rpm_diff_tol: if given, only match points within ±rpm_diff_tol
    - rolling_window: window for rolling mean smoothing
    - plot: if True, generate scatter plot of efficiency vs RPM
    
    Returns:
    - combined_df: concatenated DataFrame with efficiency calculations
    """
    
    all_runs = []
    
    for motec_file, dyno_file in zip(motec_files, dyno_files):
        # --- Load dyno ---
        print(f'Dyno_Data/{dyno_file}')
        cur_dyno_df = pd.read_csv(f'Dyno_Data/{dyno_file}', index_col=0, parse_dates=True)
        cur_dyno_df = cur_dyno_df.dropna(subset=['Axle Speed (rpm)', 'Axle Torque (Nm)'])
        
        rpm_min = cur_dyno_df["Axle Speed (rpm)"].min()
        rpm_max = cur_dyno_df["Axle Speed (rpm)"].max()
        
        # --- Load MoTeC ---
        motec_df = pd.read_csv(f'Motec_Data/CSV Export/{motec_file}', parse_dates=True, skiprows=range(0, 8), header=0 )

        # drop the units row
        motec_df = motec_df.iloc[1:].reset_index(drop=True)

        print(rpm_min, rpm_max)

        
        # Filter by RPM range and target torque
        motec_trim = motec_df[
            (motec_df["Car.Data.Motor.MotorRPM"] >= rpm_min) &
            (motec_df["Car.Data.Motor.MotorRPM"] <= rpm_max) &
            (motec_df['Car.Data.Inverter.InverterCalculatedTorque'] >= target_torque)
        ].copy()
        
        # Only keep monotonic increasing RPM
        motec_trim = motec_trim[motec_trim["Car.Data.Motor.MotorRPM"].diff() > 0]
        
        # Rolling mean smoothing
        motec_trim = motec_trim.rolling(rolling_window).mean().dropna()
        
        # --- Nearest neighbor matching ---
        dyno_features = cur_dyno_df[['Axle Speed (rpm)', 'Axle Torque (Nm)']].to_numpy()
        motec_features = motec_trim[['Car.Data.Motor.MotorRPM', 'Car.Data.Inverter.InverterCalculatedTorque']].to_numpy()
        
        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(dyno_features)
        distances, indices = nn.kneighbors(motec_features)
        
        matched_dyno = cur_dyno_df.iloc[indices.flatten()].reset_index(drop=True)
        matched_motec = motec_trim.reset_index(drop=True)
        
        # Optional RPM difference filter
        if rpm_diff_tol is not None:
            rpm_diff = np.abs(matched_dyno['Axle Speed (rpm)'] - matched_motec['Car.Data.Motor.MotorRPM'])
            keep = rpm_diff <= rpm_diff_tol
            matched_dyno = matched_dyno[keep].reset_index(drop=True)
            matched_motec = matched_motec[keep].reset_index(drop=True)
        
        # --- Compute efficiency ---
        matched_df = matched_motec.copy()
        matched_df['P_out'] = matched_dyno['Axle Torque (Nm)'] * 2*np.pi*matched_dyno['Axle Speed (rpm)']/60 / 1000
        matched_df['P_in'] = matched_motec['Car.Data.Battery.BatteryPower'] / 100000
        matched_df['eff'] = matched_df['P_out'] / matched_df['P_in']
        
        all_runs.append(matched_df)
    
    # --- Combine all runs ---
    combined_df = pd.concat(all_runs, ignore_index=True)
    combined_df = combined_df[combined_df['eff'] < 1]  # sanity filter
    
    # --- Plot ---
    if plot:
        plt.figure(figsize=(10,5))
        plt.scatter(combined_df['Car.Data.Motor.MotorRPM'], combined_df['eff'], s=5, alpha=0.7)
        plt.xlabel("RPM")
        plt.ylabel("Efficiency")
        plt.title(f"Motor Efficiency vs RPM (Torque = {target_torque} Nm)")
        plt.grid(True)
        plt.show()
    
    return combined_df

In [69]:
process_and_compute_efficiency(sorted_dyno_files[1:], motec_sorted[1:], target_torque=100, rpm_diff_tol=None, rolling_window=3, plot=True)

Dyno_Data/speedramp2_pl60_manual_dhruv.csv
661.76 1413.84


C:\Users\kafle\AppData\Local\Temp\ipykernel_33212\323662725.py:31: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  motec_df = pd.read_csv(f'Motec_Data/CSV Export/{motec_file}', parse_dates=True, skiprows=range(0, 8), header=0 )


ValueError: Found array with 0 sample(s) (shape=(0, 2)) while a minimum of 1 is required by NearestNeighbors.

In [40]:
from pathlib import Path


folder = Path("Dyno_Data")

csv_files = [f.name for f in folder.glob('*.csv')]

print(csv_files)

['speedramp10_pl140_manual_dhruv.csv', 'speedramp11_pl150_manual_dhruv.csv', 'speedramp12_pl160_maual_dhruv_failed.csv', 'speedramp13_pl167-1_manual_dhruv_failed.csv', 'speedramp1_pl50_manual_dhruv.csv', 'speedramp2_pl60_manual_dhruv.csv', 'speedramp3_pl70_manual_dhruv.csv', 'speedramp4_pl80_manual_dhruv.csv', 'speedramp5_pl90_manual_dhruv.csv', 'speedramp6_pl100_manual_dhruv.csv', 'speedramp7_pl110_manual_dhruv.csv', 'speedramp8_pl120_manual_dhruv.csv', 'speedramp9_pl130_manual_dhruv.csv']


In [49]:
import re
def extract_run_number(filename):
    return int(re.search(r"speedramp(\d+)", filename).group(1))

# filter out 13, then sort
sorted_dyno_files = sorted(
    [f for f in csv_files if extract_run_number(f) <= 12],
    key=extract_run_number
)

print(sorted_dyno_files[0])

speedramp1_pl50_manual_dhruv.csv


In [42]:
df_test = pd.read_csv(f'Dyno_Data/{sorted_dyno_files[0]}', index_col=0, parse_dates=True)
df_test

,Axle Torque (Nm),Axle Speed (rpm),Power (kW),TqC (Nm),Time (sec),Tailshaft Speed (rpm),Speed (km/h),Tacho [Rat] (rpm),Speed L (rpm),Tq L (Nm),Speed R (rpm),Axle Tq R (Nm),Comments
Run Name,,,,,,,,,,,,,
MUREV2025.0028,142.5,756.52,11.249,43.506,1.30,2958.0,58.025,2469.1,759.0,97.5,754.1,45.0,NaN
MUREV2025.0028,142.5,755.80,11.252,43.562,1.35,2955.2,57.967,2466.7,758.2,98.0,753.4,44.5,NaN
MUREV2025.0028,142.5,756.00,11.229,43.500,1.40,2956.0,57.931,2465.2,758.4,98.5,753.6,44.0,NaN
MUREV2025.0028,142.5,756.52,11.226,43.498,1.45,2958.0,57.917,2464.6,759.0,98.5,754.0,44.0,NaN
MUREV2025.0028,142.0,757.60,11.200,43.380,1.50,2962.2,57.938,2465.5,760.2,98.5,755.0,43.5,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
MUREV2025.0028,149.0,1418.40,22.021,45.523,14.90,5545.9,108.554,4619.4,1418.8,39.5,1418.0,109.5,NaN
MUREV2025.0028,149.0,1414.56,21.996,45.512,14.95,5530.9,108.454,4615.1,1415.8,39.5,1413.3,109.5,NaN
MUREV2025.0028,148.5,1403.56,21.851,45.357,15.00,5487.9,108.108,4600.4,1407.2,39.0,1399.9,109.5,NaN


In [43]:
folder = Path("Motec_Data/CSV Export")

csv_motec_files = [f.name for f in folder.glob('*.csv')]

print(csv_motec_files)

['dryno_morning_torquemapping_withshudderissue_1.csv', 'dryno_morning_torquemapping_withshudderissue_2.csv', 'dryno_morning_torquemapping_withshudderissue_3.csv', 'dryno_morning_torquemapping_withshudderissue_4.csv', 'dryno_morning_torquemapping_withshudderissue_5.csv', 'dryno_morning_torquemapping_withshudderissue_6.csv', 'dryno_morning_torquemapping_withshudderissue_7.csv', 'dyno_arvo_shudderfix_andlowrpmspeedramp_1.csv', 'dyno_arvo_shudderfix_andlowrpmspeedramp_10.csv', 'dyno_arvo_shudderfix_andlowrpmspeedramp_11.csv', 'dyno_arvo_shudderfix_andlowrpmspeedramp_12.csv', 'dyno_arvo_shudderfix_andlowrpmspeedramp_2.csv', 'dyno_arvo_shudderfix_andlowrpmspeedramp_3.csv', 'dyno_arvo_shudderfix_andlowrpmspeedramp_4.csv', 'dyno_arvo_shudderfix_andlowrpmspeedramp_5.csv', 'dyno_arvo_shudderfix_andlowrpmspeedramp_6.csv', 'dyno_arvo_shudderfix_andlowrpmspeedramp_7.csv', 'dyno_arvo_shudderfix_andlowrpmspeedramp_8.csv', 'dyno_arvo_shudderfix_andlowrpmspeedramp_9.csv']


In [47]:
def extract_num(f):
    return int(re.search(r"_(\d+)\.csv", f).group(1))

# filter + sort
motec_sorted = sorted(
    [f for f in csv_motec_files if "dyno_arvo" in f],
    key=extract_num
)

print(motec_sorted)

['dyno_arvo_shudderfix_andlowrpmspeedramp_1.csv', 'dyno_arvo_shudderfix_andlowrpmspeedramp_2.csv', 'dyno_arvo_shudderfix_andlowrpmspeedramp_3.csv', 'dyno_arvo_shudderfix_andlowrpmspeedramp_4.csv', 'dyno_arvo_shudderfix_andlowrpmspeedramp_5.csv', 'dyno_arvo_shudderfix_andlowrpmspeedramp_6.csv', 'dyno_arvo_shudderfix_andlowrpmspeedramp_7.csv', 'dyno_arvo_shudderfix_andlowrpmspeedramp_8.csv', 'dyno_arvo_shudderfix_andlowrpmspeedramp_9.csv', 'dyno_arvo_shudderfix_andlowrpmspeedramp_10.csv', 'dyno_arvo_shudderfix_andlowrpmspeedramp_11.csv', 'dyno_arvo_shudderfix_andlowrpmspeedramp_12.csv']


In [70]:
print(motec_sorted[0])
df_test = pd.read_csv(f'Motec_Data/CSV Export/{motec_sorted[1]}', index_col=0, parse_dates=True, skiprows=range(0, 8), header=0 )

# drop the units row
df_test = df_test.iloc[1:].reset_index(drop=True)

df_test

dyno_arvo_shudderfix_andlowrpmspeedramp_1.csv


C:\Users\kafle\AppData\Local\Temp\ipykernel_33212\4218521285.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_test = pd.read_csv(f'Motec_Data/CSV Export/{motec_sorted[1]}', index_col=0, parse_dates=True, skiprows=range(0, 8), header=0 )


,Car.DigitalInput.MSS.PDOC.Pin,Car.DigitalInput.MSS.BMS.Pin,Car.DigitalInput.MSS.BSPD.Pin,Car.DigitalInput.MSS.IMD.Pin,Car.Data.Driver.ThrottlePressure,Car.Data.Motor.MotorRPM,Car.AnalogInput.BMS.CellTempASensor,Car.AnalogInput.BMS.CellTempBSensor,Car.AnalogInput.BMS.CellTempCSensor,Car.Data.Battery.PackSOC,...,Car.Data.IMU.Rear.AccelX,Car.Data.IMU.Rear.GyroZ,Car.Data.IMU.Rear.GyroY,Car.Data.IMU.Rear.GyroX,Car.Data.IMU.Rear.VelD,Car.Data.IMU.Rear.VelE,Car.Data.IMU.Rear.VelN,Car.DigitalInput.WheelSpeed.WheelSpeedFrontLeft,Car.DigitalInput.WheelSpeed.WheelSpeedFrontRight,Car.Data.Inverter.InverterCalculatedTorque
0,1.0,1.0,1.0,1.0,0.0,0.0,1.69319,0.001012,1.68935,49.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1.0,1.0,1.0,1.0,0.0,0.0,1.68804,0.001764,1.68522,49.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1.0,1.0,1.0,1.0,0.0,0.0,1.68288,0.002515,1.68109,49.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1.0,1.0,1.0,1.0,0.0,0.0,1.67773,0.003266,1.67696,49.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1.0,1.0,1.0,1.0,0.0,0.0,1.67257,0.004017,1.67283,49.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44432,0.0,0.0,0.0,0.0,0.0,0.0,1.70531,0.000434,1.70052,49.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
44433,0.0,0.0,0.0,0.0,0.0,0.0,1.70531,0.000434,1.70052,49.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
44434,0.0,0.0,0.0,0.0,0.0,0.0,1.70531,0.000434,1.70052,49.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
44435,0.0,0.0,0.0,0.0,0.0,0.0,1.70531,0.000434,1.70052,49.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
